# Лабораторная работа № 3. Способы кодирования данных

Курс: «Теория информационных процессов и систем». Выполнены задания № 2, 4, 7, 8 и 10.

Ноутбук можно открыть в Jupyter или Google Colab и выполнить все ячейки по порядку. Если в ячейке про RSA нет модуля `cryptography`, выполните `%pip install cryptography` в отдельной ячейке и повторите запуск.

## Задание 2. RLE

Функция идёт по строке и считает, сколько раз подряд встретился текущий символ. При смене символа она сохраняет пару `(символ, количество)` и начинает новую серию. Последнюю серию нужно записать после цикла. Пустая строка даёт пустой список.

В строке `AAAABBCCCA` четыре серии. Последняя `A` относится к новой серии, потому что между ней и первыми буквами `A` стоят другие символы.

In [1]:
def rle_encode(text):
    if not text:
        return []

    runs = []
    current = text[0]
    count = 1

    for char in text[1:]:
        if char == current:
            count += 1
        else:
            runs.append((current, count))
            current = char
            count = 1

    runs.append((current, count))
    return runs


for text in ("AAAABBCCCA", "ABCDEF", ""):
    print(f"{text!r}: {rle_encode(text)}")

'AAAABBCCCA': [('A', 4), ('B', 2), ('C', 3), ('A', 1)]
'ABCDEF': [('A', 1), ('B', 1), ('C', 1), ('D', 1), ('E', 1), ('F', 1)]
'': []


В примере `ABCDEF` каждая серия состоит из одного символа. При записи количества для каждой серии размер данных в таком случае может увеличиться. Число пар само по себе не показывает, сколько байтов занял бы результат в файле.

## Задание 4. Код Грея для чисел от 0 до 15

Для числа `n` берём XOR его двоичной записи и той же записи со сдвигом вправо на один бит: `n ^ (n >> 1)`. Выводим результат в четырёх разрядах. Два соседних числа в полученной последовательности отличаются ровно одним битом. Например, у 7 и 8 обычные двоичные записи `0111` и `1000`, а коды Грея имеют вид `0100` и `1100`.

In [2]:
def gray_code(n):
    return n ^ (n >> 1)


print(f"{'Число':>5}  {'Двоичный':>8}  {'Код Грея':>8}")

for n in range(16):
    binary = f"{n:04b}"
    gray = f"{gray_code(n):04b}"
    print(f"{n:>5}  {binary:>8}  {gray:>8}")

Число  Двоичный  Код Грея
    0      0000      0000
    1      0001      0001
    2      0010      0011
    3      0011      0010
    4      0100      0110
    5      0101      0111
    6      0110      0101
    7      0111      0100
    8      1000      1100
    9      1001      1101
   10      1010      1111
   11      1011      1110
   12      1100      1010
   13      1101      1011
   14      1110      1001
   15      1111      1000


## Задание 7. Генерация ключей RSA

`generate_private_key()` создаёт закрытый ключ, а `public_key()` возвращает соответствующий открытый. Размер ключа задаёт длину модуля `n` в битах; `65537` задаёт открытую экспоненту `e`. При повторном запуске получится новая пара. В выводе есть только открытые параметры и открытый ключ в текстовом формате PEM.

[Документация cryptography: RSA](https://cryptography.io/en/latest/hazmat/primitives/asymmetric/rsa/)

In [3]:
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa

private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()
public_numbers = public_key.public_numbers()

public_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
)

print("Длина модуля в битах:", public_numbers.n.bit_length())
print("Открытая экспонента:", public_numbers.e)
print(public_pem.decode("ascii"), end="")

Длина модуля в битах: 2048
Открытая экспонента: 65537
-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAtDETsbd0vkslchWGMEBv
Qk4BddzLX25xofmQFtb9ihU9N+l45tnciN8W3zDkXWzksp8JIyfqQsQPatqwaFa0
w8EJURImWrbMsP7FN6LVzHOQx4h19OVUWqS/d3jc8AIlBqKXF87ycY2+M6QtWooi
HDxbuseKzbbEOFA5rc8pP/rzopGbb5MQVUKouqdEGtEBb4Wb5bFRJdgK1Qwc/Oi3
GuVoZzGQ8/fWfG7nExyeg41/xmUxqzLO9KYVHs/ZExIJcMWHigEIvO2kTy9u+8DX
Qk1JqkHc2ZZ/v9ISptkBSMQNGgCPh8Z+gsKr5gZp7gWV7tjljaNaKkN0pgNgMv3k
cwIDAQAB
-----END PUBLIC KEY-----


## Задание 8. XOR-кодирование

Строку сначала переводим в байты UTF-8. Каждый байт складываем с одним и тем же байтом ключа по операции XOR. Повторное применение с тем же ключом восстанавливает исходные байты: `(x ^ k) ^ k == x`. Промежуточный результат показываем в шестнадцатеричной записи, потому что полученные байты могут не образовывать текст UTF-8.

У ключа из одного байта всего 256 вариантов. Такой пример показывает обратимость XOR; для защиты данных одного повторяющегося байта недостаточно.

In [4]:
def xor_bytes(data, key):
    return bytes(value ^ key for value in data)


message = "Привет"
key = 42
encoded = xor_bytes(message.encode("utf-8"), key)
decoded = xor_bytes(encoded, key).decode("utf-8")

print("Исходная строка:", message)
print("Байты после XOR:", encoded.hex())
print("После повторного XOR:", decoded)

Исходная строка: Привет
Байты после XOR: fab5fbaafa92fa98fa9ffba8
После повторного XOR: Привет


## Задание 10. MD5 и SHA-256

Обе функции принимают байты и выдают хеш фиксированной длины. У MD5 это 128 бит (32 шестнадцатеричных знака), у SHA-256 это 256 бит (64 знака). По хешу исходное сообщение не декодируют. Коллизия возникает, когда у разных сообщений совпадают хеши. Для MD5 известны практические способы построения коллизий, поэтому его не используют там, где нужно защититься от подмены.

Сравним `hello` и `Hello`: в UTF-8 их записи отличаются одним битом. В примере меняются оба хеша. Повторный расчёт для `hello` даёт прежний результат.

[Документация Python: hashlib](https://docs.python.org/3/library/hashlib.html)

In [5]:
from hashlib import md5, sha256

for message in ("hello", "Hello", "hello"):
    data = message.encode("utf-8")
    md5_hash = md5(data, usedforsecurity=False).hexdigest()
    sha256_hash = sha256(data).hexdigest()

    print(f"{message!r}")
    print(f"MD5:     {md5_hash} ({len(md5_hash) * 4} бит)")
    print(f"SHA-256: {sha256_hash} ({len(sha256_hash) * 4} бит)")
    print()

'hello'
MD5:     5d41402abc4b2a76b9719d911017c592 (128 бит)
SHA-256: 2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824 (256 бит)

'Hello'
MD5:     8b1a9953c4611296a827abf8c47804d7 (128 бит)
SHA-256: 185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969 (256 бит)

'hello'
MD5:     5d41402abc4b2a76b9719d911017c592 (128 бит)
SHA-256: 2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824 (256 бит)

